# 05 — So sánh prediction đa phương thức và ResNet trên dev/eval

Nguồn chính thức: run 04 `20261110-074559`. Không yêu cầu tái lập H100 cũ.
Chỉ đọc prediction và nhãn val, không GPU, ảnh, Qwen, inference ResNet hay dữ liệu test.
Test 04 đã được bật trước phân tích này: không dùng 05 để đổi full, seed, ngưỡng hay cấu hình 04.
Ensemble, late fusion và top-k chỉ phân tích eval, không được chấm test sau đó.
Mọi metric báo cáo được tính lại từ prediction, không chép bảng tổng hợp nguồn.


## 1. Đồng bộ code
Cell giống 02–04. Colab cần code trên main; Modal dùng snapshot local.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition"
)
REPO_URL = (
    "https://github.com/minh-1105/"
    "Explainable-Contextual-Emotion-Recognition-via-"
    "Text-Guided-Chain-of-Thought-Reasoning.git"
)
CODE_ROOT = Path("/content/contextual-emotion-reasoning")

assert DRIVE_ROOT.is_dir(), f"Không tìm thấy project trên Drive: {DRIVE_ROOT}"

if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")],
    check=True,
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Drive root:", DRIVE_ROOT)
print("Code root :", CODE_ROOT)

## 2. Cấu hình và run mới
Không resume; threshold grids lấy nguyên từ config 04. Bootstrap 2.000 lần theo ảnh.

In [ ]:
import json
import subprocess
from pathlib import Path
from hashlib import sha256
from src.evaluation.multimodal_comparison import ComparisonStudy, SOURCE_RUN, RESNET_RUN
GIT_COMMIT = subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()
study = ComparisonStudy(DRIVE_ROOT, GIT_COMMIT, source_run=SOURCE_RUN, resnet_run=RESNET_RUN)
CONFIG, RUN_DIR = study.config, study.out
paths = sorted((CODE_ROOT / "src").rglob("*.py")) + [CODE_ROOT / "notebooks/05_Compare_Multimodal_Baselines.ipynb", CODE_ROOT / "requirements.txt"]
hashes = {p.relative_to(CODE_ROOT).as_posix(): sha256(p.read_bytes()).hexdigest() for p in paths}
(RUN_DIR / "code_snapshot.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "files": hashes}, indent=2), encoding="utf-8")
print("Run kết quả:", RUN_DIR)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


## 3. Nạp và kiểm tra nguồn
Hash mọi file đọc; assert 26 nhãn, nhãn cache val, thứ tự từng người và ảnh dev/eval tách biệt. Không materialize mảng feature.

In [ ]:
study.load_sources()

## 4–5. Tái lập bắt buộc
Mọi biến thể × seed × grid × part của 04 và ba ResNet phải khớp metric đã lưu (atol 1e-9). Chọn lại ngưỡng trên dev, lệch là dừng. Không so H100.

In [ ]:
study.reproduce()

## 6. Bảng chính trên eval
Mean ± SD qua seed, kèm từng seed. Extended là lưới chính, legacy đối chiếu. Constant top-k chọn trên dev; AP dùng prevalence score cố định theo lớp.

In [ ]:
study.main_table()
for row in study.rows:
    if row["grid"] == "extended" and (row.get("summary") or row["model"] in ("person_only", "context_only", "person_context", "constant_top_k")):
        print(row["model"], "mAP:", row["map"], "macro-F1:", row["macro_f1"])

## 7. Phân tích bổ sung chỉ trên eval
Ensemble trung bình xác suất full qua seed. Late fusion chọn w theo dev mAP, rồi threshold theo dev F1. Mỗi model chọn k bằng cùng luật dev macro-F1. Khóa mọi tham số trước metric eval; không chấm test.

In [ ]:
study.supplementary()

## 8. Bootstrap ghép cặp theo ảnh
Mỗi seed: full − mỗi ablation, ResNet A/B/C. Thêm ensemble − ResNet C và late fusion − ResNet C. Log và lưu sau từng metric. CI chứa 0: “không phát hiện đóng góp rõ”, không nói “không có tác dụng”. Không gộp p-value.

In [ ]:
study.bootstrap()

## 9–10. Theo lớp, số người và ví dụ lỗi
Đếm người từ dev ∪ eval theo image_id: nhóm 1 / 2 / 3–4 / ≥5. AP delta full trung bình ba seed − ResNet C. Ví dụ lỗi dùng seed đầu tiên cố định trong config, xếp chênh lệch sample-F1 rồi sample_id, chỉ lấy đúng chiều hơn/thua.

In [ ]:
study.diagnostics()

## 11. Hình và kiểm tra bàn giao
Biểu đồ metric ± SD seed, forest plot bootstrap, AP delta và metric theo nhóm người.

In [ ]:
study.figures()
required = ["config.json", "environment.txt", "code_snapshot.json", "source_artifacts.json", "reproduction_checks.json", "metrics.csv", "metrics.json", "per_class_metrics.csv", "per_class_ap_delta.csv", "bootstrap.json", "people_group_metrics.json", "selected_parameters.json", "error_examples.json", "predictions_dev.npz", "predictions_eval.npz", "figures/comparison.png", "figures/bootstrap.png", "figures/ap_delta.png", "figures/people_groups.png"]
assert all((RUN_DIR / name).is_file() for name in required)
print("Đủ artifact bàn giao:", RUN_DIR)

## 12. Đọc kết quả và giới hạn

Đọc reproduction_checks.json trước, sau đó metrics.csv, bootstrap.json, per_class_ap_delta.csv và hình.
Kết luận cụ thể chỉ viết sau khi chạy và đọc artifact; không tuyên bố cải thiện nếu CI chứa 0.

- ResNet một seed, checkpoint chọn trên toàn bộ PAMI val gồm eval.
- Feature 04 mã hóa lại, không yêu cầu bằng H100 từng bit.
- Ensemble/late fusion/top-k chỉ phân tích eval; không chấm test sau đó.
- Không chỉnh cấu hình 04 theo kết quả này vì test đã bật.
- Hash hiện tại không tự chứng minh lịch sử trước thời điểm tạo manifest.
